In [1]:
import torch
from termcolor import colored
from transformers import AutoModelForCausalLM, AutoTokenizer




In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.1"
).to('cuda')

tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1")


In [2]:
tokenizer = AutoTokenizer.from_pretrained("intfloat/e5-mistral-7b-instruct")
model = AutoModelForCausalLM.from_pretrained(
  "intfloat/e5-mistral-7b-instruct"
).to('cuda')

/usr/lib/python3/dist-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/usr/lib/python3/dist-packages/torchvision/image.so: undefined symbol: _ZN3c104cuda9SetDeviceEi'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
/usr/lib/python3/dist-packages/torchvision/datapoints/__init__.py:12: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and you can also check out https://github.com/pytorch/vision/issues/7319 to learn more about the APIs that we suspect might involve future changes. You can silen

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of MistralForCausalLM were not initialized from the model checkpoint at intfloat/e5-mistral-7b-instruct and are newly initialized: ['lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [3]:
tokenizer.chat_template = "{% if not add_generation_prompt is defined %}{% set add_generation_prompt = false %}{% endif %}{% for message in messages %}{{'<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"


def generate_last_token_embeddings(question):
    messages = [
        {
            "role": "user",
            "content": question,
        },
    ]
    encodeds = tokenizer.apply_chat_template(messages, return_tensors="pt")
    model_inputs = encodeds.to("cuda")
    with torch.no_grad():
        return model(model_inputs).logits[:, -1, :]

def get_similarities(questions, answers):
    for question in questions:
        for answer in answers:
            q_embedding, a_embedding = (
                generate_last_token_embeddings(question),
                generate_last_token_embeddings(answer),
            )
            similarity = compute_cosine_similarity(q_embedding, a_embedding)
            print(colored(f"question: {question} and ans: {answer}", "green"))
            print(colored(f"result: {similarity}", "blue"))
            
def compute_cosine_similarity(vec1, vec2):
    cos = torch.nn.CosineSimilarity(dim=1, eps=1e-6)
    return cos(vec1, vec2)

questions = ["Where is the headquarters of OpenAI?", "What is GPU?"]
answers = [
    "This is ECE 645 class.",
    "A graphics processing unit (GPU) is an electronic circuit that can perform mathematical calculations quickly",
]
get_similarities(questions, answers)

question: Where is the headquarter of OpenAI? and ans: This is ECE 645 class.
result: tensor([0.7501], device='cuda:0')
question: Where is the headquarter of OpenAI? and ans: A graphics processing unit (GPU) is an electronic circuit that can perform mathematical calculations quickly
result: tensor([0.7552], device='cuda:0')
question: What is GPU? and ans: This is ECE 645 class.
result: tensor([0.7982], device='cuda:0')
question: What is GPU? and ans: A graphics processing unit (GPU) is an electronic circuit that can perform mathematical calculations quickly
result: tensor([0.9268], device='cuda:0')


In [6]:
messages=  [
        {
            "role": "user",
            "content": "what is a GPU?",
        },
    ]

encodeds = tokenizer.apply_chat_template(messages, return_tensors="pt")

print(encodeds)
print(tokenizer.decode(encodeds[0]))
print(encodeds.shape)

tensor([[  523, 28766,   321, 28730,  2521, 28766, 28767,  1838,    13,  6802,
           349,   264, 28475, 28804, 28789, 28766,   321, 28730,   416, 28766,
         28767,    13]])
<|im_start|>user
what is a GPU?<|im_end|>

torch.Size([1, 22])


In [7]:
model_inputs = encodeds.to("cuda")
with torch.no_grad():
        logits =  model(model_inputs).logits

In [7]:
print(logits.shape)

torch.Size([1, 22, 32000])


In [8]:
for param in model.named_parameters():
    if 'embed' in param[0]:
        WE = param[1]

print(WE.shape)

torch.Size([32000, 4096])


In [16]:
import numpy as np

prod = WE.T @ WE
print(np.round(prod[:5,:5].detach().cpu().numpy(),1))

[[ 0.2 -0.  -0.  -0.  -0. ]
 [-0.   0.2  0.  -0.  -0. ]
 [-0.   0.   0.2  0.   0. ]
 [-0.  -0.   0.   0.3  0. ]
 [-0.  -0.   0.   0.   0.2]]


In [17]:
for param in model.named_parameters():
    print(param[0])

model.embed_tokens.weight
model.layers.0.self_attn.q_proj.weight
model.layers.0.self_attn.k_proj.weight
model.layers.0.self_attn.v_proj.weight
model.layers.0.self_attn.o_proj.weight
model.layers.0.mlp.gate_proj.weight
model.layers.0.mlp.up_proj.weight
model.layers.0.mlp.down_proj.weight
model.layers.0.input_layernorm.weight
model.layers.0.post_attention_layernorm.weight
model.layers.1.self_attn.q_proj.weight
model.layers.1.self_attn.k_proj.weight
model.layers.1.self_attn.v_proj.weight
model.layers.1.self_attn.o_proj.weight
model.layers.1.mlp.gate_proj.weight
model.layers.1.mlp.up_proj.weight
model.layers.1.mlp.down_proj.weight
model.layers.1.input_layernorm.weight
model.layers.1.post_attention_layernorm.weight
model.layers.2.self_attn.q_proj.weight
model.layers.2.self_attn.k_proj.weight
model.layers.2.self_attn.v_proj.weight
model.layers.2.self_attn.o_proj.weight
model.layers.2.mlp.gate_proj.weight
model.layers.2.mlp.up_proj.weight
model.layers.2.mlp.down_proj.weight
model.layers.2.inp

In [20]:
for param in model.named_parameters():
    if 'layers.0' in param[0]:
        print(param[0],'\n')
        print('Shape: ', param[1].shape)
        

model.layers.0.self_attn.q_proj.weight 

Shape:  torch.Size([4096, 4096])
model.layers.0.self_attn.k_proj.weight 

Shape:  torch.Size([1024, 4096])
model.layers.0.self_attn.v_proj.weight 

Shape:  torch.Size([1024, 4096])
model.layers.0.self_attn.o_proj.weight 

Shape:  torch.Size([4096, 4096])
model.layers.0.mlp.gate_proj.weight 

Shape:  torch.Size([14336, 4096])
model.layers.0.mlp.up_proj.weight 

Shape:  torch.Size([14336, 4096])
model.layers.0.mlp.down_proj.weight 

Shape:  torch.Size([4096, 14336])
model.layers.0.input_layernorm.weight 

Shape:  torch.Size([4096])
model.layers.0.post_attention_layernorm.weight 

Shape:  torch.Size([4096])
